In [ ]:
# =====================================================================
# FINAL SINGLE-CELL COLAB SIMULATION
# Symmetry-Preserving Transformer-Based Shape Prediction
# + Neural Shape Control
# + Whole-Arm Object Caging
#
# 50-link planar 2D hyper-redundant robot
# Kinematic-level, frictionless robot-object interaction
# 5 independent scenes
# 4 static obstacles + 1 moving obstacle
#
# OUTPUTS:
#   - scene_1_hyper_redundant_caging.mp4
#   - ...
#   - scene_5_hyper_redundant_caging.mp4
#   - scene_results.csv
#   - summary_mean_std.csv
#   - Table3_Implementation_Hyperparameters.csv
#
# Run this entire cell once in Google Colab.
# =====================================================================

# ============================
# 0. IMPORTS AND DRIVE
# ============================

import os
import math
import time
import random
import heapq
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from matplotlib.patches import Rectangle, Circle

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass


# ============================
# 1. REPRODUCIBILITY
# ============================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)


# ============================
# 2. OUTPUT DIRECTORY
# ============================

DRIVE_DIR = "/content/drive/MyDrive/HyperRedundantRobot_Final"

if os.path.exists("/content/drive/MyDrive"):
    OUTPUT_DIR = DRIVE_DIR
else:
    OUTPUT_DIR = "/content/HyperRedundantRobot_Final"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory:", OUTPUT_DIR)


# ============================
# 3. GLOBAL CONFIGURATION
# ============================

CFG = {

    # ------------------------
    # Robot
    # ------------------------
    "N_LINKS": 50,
    "LINK_LENGTH": 0.30,
    "ROBOT_RADIUS": 0.055,

    "BASE": np.array([-4.45, -3.25], dtype=float),

    # ------------------------
    # Cage
    # ------------------------
    # Last 12 links form near-closed cage
    "N_CAGE_LINKS": 12,
    "CAGE_GAP_ANGLE": 0.50,

    "OBJECT_RADIUS": 0.28,

    # ------------------------
    # Workspace
    # ------------------------
    "X_MIN": -5.0,
    "X_MAX": 5.0,
    "Y_MIN": -4.0,
    "Y_MAX": 4.0,

    # ------------------------
    # Safety
    # ------------------------
    "D_SAFE": 0.12,
    "D_SELF": 0.10,

    # A larger margin is used for object path planning.
    "OBJECT_PATH_MARGIN": 0.18,

    # ------------------------
    # Controller
    # ------------------------
    "CONTROL_GAIN": 0.68,
    "MAX_POINT_STEP": 0.22,

    # ------------------------
    # Shape refinement
    # ------------------------
    "SHAPE_REFINE_ITERS": 45,
    "ACTUAL_CONSTRAINT_ITERS": 16,

    # ------------------------
    # Experiment
    # ------------------------
    "N_SCENES": 5,
    "WARMUP_FRAMES": 36,
    "TRANSPORT_STEPS": 125,

    # one extra control frame can be inserted
    "MAX_TOTAL_FRAMES": 220,

    # ------------------------
    # Video
    # ------------------------
    "FPS": 12,
    "VIDEO_DPI": 125,
    "SAVE_VIDEO": True,

    # ------------------------
    # Transformer
    # ------------------------
    "D_MODEL": 64,
    "NHEAD": 4,
    "NUM_ENCODER_LAYERS": 2,
    "DIM_FEEDFORWARD": 128,
    "DROPOUT": 0.05,

    "TRANSFORMER_EPOCHS": 28,
    "TRANSFORMER_BATCH_SIZE": 32,
    "TRANSFORMER_LR": 8e-4,

    # ------------------------
    # Neural controller
    # ------------------------
    "NSC_EPOCHS": 18,
    "NSC_BATCH_SIZE": 256,
    "NSC_LR": 1e-3,
}


# Derived quantities
N = CFG["N_LINKS"]
L = CFG["LINK_LENGTH"]

N_CAGE = CFG["N_CAGE_LINKS"]
N_STEM = N - N_CAGE

GAP = CFG["CAGE_GAP_ANGLE"]

CAGE_DELTA = (2.0 * np.pi - GAP) / N_CAGE

# Radius chosen so every cage segment has exact length LINK_LENGTH
CAGE_RADIUS = L / (2.0 * np.sin(CAGE_DELTA / 2.0))

CFG["CAGE_RADIUS"] = float(CAGE_RADIUS)

print(f"Stem links        : {N_STEM}")
print(f"Cage links        : {N_CAGE}")
print(f"Cage radius       : {CAGE_RADIUS:.4f}")
print(f"Total robot length: {N * L:.3f}")


# ============================
# 4. SCENES
# ============================

# rectangle format:
# (cx, cy, width, height)

SCENES = [

    {
        "name": "Scene 1 - Sparse",
        "object_start": np.array([-0.80, -1.15]),
        "goal": np.array([3.55, 2.55]),

        "obstacles": [
            (-2.30,  1.60, 0.60, 0.75),
            ( 1.15,  1.10, 0.70, 0.80),
            ( 1.00, -2.25, 0.85, 0.55),
            (-3.95,  0.15, 0.55, 0.65),
        ],

        "moving_obstacle": {
            "cx": 2.20,
            "cy": -0.30,
            "w": 0.55,
            "h": 0.55,
            "amp": 0.60,
            "period": 80,
            "axis": "y",
        },
    },

    {
        "name": "Scene 2 - Narrow Passage",
        "object_start": np.array([-0.65, -1.20]),
        "goal": np.array([3.55, 2.60]),

        "obstacles": [
            (-1.90,  1.15, 0.65, 1.25),
            ( 0.25,  1.20, 0.65, 1.25),
            ( 1.75, -1.55, 0.75, 0.85),
            (-3.70,  0.10, 0.55, 0.60),
        ],

        "moving_obstacle": {
            "cx": 2.55,
            "cy": 0.20,
            "w": 0.50,
            "h": 0.60,
            "amp": 0.55,
            "period": 75,
            "axis": "y",
        },
    },

    {
        "name": "Scene 3 - Dense",
        "object_start": np.array([-0.80, -1.35]),
        "goal": np.array([3.65, 2.50]),

        "obstacles": [
            (-2.45,  1.35, 0.70, 0.80),
            (-0.10,  1.35, 0.70, 0.75),
            ( 1.45, -1.20, 0.75, 0.80),
            ( 2.45,  1.05, 0.70, 0.75),
        ],

        "moving_obstacle": {
            "cx": 0.45,
            "cy": -0.15,
            "w": 0.55,
            "h": 0.55,
            "amp": 0.70,
            "period": 90,
            "axis": "x",
        },
    },

    {
        "name": "Scene 4 - Asymmetric",
        "object_start": np.array([-0.95, -1.20]),
        "goal": np.array([3.55, 2.45]),

        "obstacles": [
            (-2.65,  1.70, 0.70, 0.70),
            (-1.15,  0.60, 0.60, 0.70),
            ( 1.60, -1.60, 0.75, 0.80),
            ( 2.45,  0.50, 0.60, 0.85),
        ],

        "moving_obstacle": {
            "cx": 0.60,
            "cy": 1.75,
            "w": 0.55,
            "h": 0.55,
            "amp": 0.65,
            "period": 85,
            "axis": "x",
        },
    },

    {
        "name": "Scene 5 - Highly Cluttered",
        "object_start": np.array([-0.85, -1.35]),
        "goal": np.array([3.55, 2.55]),

        "obstacles": [
            (-2.35,  1.50, 0.70, 0.75),
            (-0.20,  1.15, 0.70, 0.75),
            ( 1.40, -1.55, 0.75, 0.70),
            ( 2.65,  0.90, 0.65, 0.80),
        ],

        "moving_obstacle": {
            "cx": 0.85,
            "cy": -0.10,
            "w": 0.55,
            "h": 0.55,
            "amp": 0.75,
            "period": 78,
            "axis": "y",
        },
    },
]


# ============================
# 5. BASIC GEOMETRY
# ============================

def rect_bounds(rect):
    cx, cy, w, h = rect
    return (
        cx - w / 2.0,
        cx + w / 2.0,
        cy - h / 2.0,
        cy + h / 2.0
    )


def moving_rect(scene, frame):
    m = scene["moving_obstacle"]

    phase = 2.0 * np.pi * frame / m["period"]

    cx = m["cx"]
    cy = m["cy"]

    if m["axis"] == "x":
        cx += m["amp"] * np.sin(phase)
    else:
        cy += m["amp"] * np.sin(phase)

    return (cx, cy, m["w"], m["h"])


def point_rect_signed_distance(p, rect):
    """
    Positive outside rectangle.
    Negative inside rectangle.
    """
    cx, cy, w, h = rect

    dx = abs(p[0] - cx) - w / 2.0
    dy = abs(p[1] - cy) - h / 2.0

    outside = np.linalg.norm(
        [max(dx, 0.0), max(dy, 0.0)]
    )

    inside = min(max(dx, dy), 0.0)

    return outside + inside


def point_rect_escape_vector(p, rect):
    """
    Unit vector pointing away from rectangle.
    """
    cx, cy, w, h = rect

    xmin, xmax, ymin, ymax = rect_bounds(rect)

    nearest = np.array([
        np.clip(p[0], xmin, xmax),
        np.clip(p[1], ymin, ymax)
    ])

    v = p - nearest

    nv = np.linalg.norm(v)

    if nv > 1e-9:
        return v / nv

    # Point inside rectangle
    candidates = [
        (abs(p[0] - xmin), np.array([-1.0, 0.0])),
        (abs(xmax - p[0]), np.array([ 1.0, 0.0])),
        (abs(p[1] - ymin), np.array([0.0, -1.0])),
        (abs(ymax - p[1]), np.array([0.0,  1.0])),
    ]

    return min(candidates, key=lambda x: x[0])[1]


def segment_rect_distance(a, b, rect, samples=9):
    ts = np.linspace(0.0, 1.0, samples)

    vals = []

    for t in ts:
        p = (1.0 - t) * a + t * b
        vals.append(point_rect_signed_distance(p, rect))

    return float(np.min(vals))


def min_link_obstacle_clearance(points, obstacles):
    dmin = np.inf

    for i in range(len(points) - 1):
        a = points[i]
        b = points[i + 1]

        for rect in obstacles:
            d = segment_rect_distance(a, b, rect)
            dmin = min(dmin, d)

    return float(dmin)


def object_obstacle_clearance(center, radius, obstacles):
    dmin = np.inf

    for rect in obstacles:
        d = point_rect_signed_distance(center, rect) - radius
        dmin = min(dmin, d)

    return float(dmin)


def segments_intersect(p1, p2, q1, q2):
    def orient(a, b, c):
        return np.cross(b - a, c - a)

    o1 = orient(p1, p2, q1)
    o2 = orient(p1, p2, q2)
    o3 = orient(q1, q2, p1)
    o4 = orient(q1, q2, p2)

    return (
        (o1 * o2 < 0.0) and
        (o3 * o4 < 0.0)
    )


def self_intersections(points):
    count = 0

    nseg = len(points) - 1

    for i in range(nseg):

        for j in range(i + 3, nseg):

            if abs(i - j) <= 1:
                continue

            if segments_intersect(
                points[i],
                points[i + 1],
                points[j],
                points[j + 1]
            ):
                count += 1

    return count


def max_backbone_curvature(points):
    vals = []

    for i in range(1, len(points) - 1):
        a = points[i] - points[i - 1]
        b = points[i + 1] - points[i]

        na = np.linalg.norm(a)
        nb = np.linalg.norm(b)

        if na < 1e-9 or nb < 1e-9:
            continue

        c = np.clip(
            np.dot(a, b) / (na * nb),
            -1.0,
            1.0
        )

        ang = np.arccos(c)

        vals.append(ang)

    if not vals:
        return 0.0

    return float(np.max(vals))


# ============================
# 6. A* PATH PLANNING
# ============================

GRID_RES = 0.12


def is_point_free(p, obstacles, margin):
    if p[0] < CFG["X_MIN"] + margin:
        return False

    if p[0] > CFG["X_MAX"] - margin:
        return False

    if p[1] < CFG["Y_MIN"] + margin:
        return False

    if p[1] > CFG["Y_MAX"] - margin:
        return False

    for r in obstacles:
        if point_rect_signed_distance(p, r) < margin:
            return False

    return True


def world_to_grid(p):
    gx = int(round((p[0] - CFG["X_MIN"]) / GRID_RES))
    gy = int(round((p[1] - CFG["Y_MIN"]) / GRID_RES))

    return gx, gy


def grid_to_world(g):
    x = CFG["X_MIN"] + g[0] * GRID_RES
    y = CFG["Y_MIN"] + g[1] * GRID_RES

    return np.array([x, y], dtype=float)


def astar_path(start, goal, obstacles, margin):

    s = world_to_grid(start)
    g = world_to_grid(goal)

    open_heap = [(0.0, s)]

    came = {}
    gcost = {s: 0.0}

    moves = [
        (-1,  0, 1.0),
        ( 1,  0, 1.0),
        ( 0, -1, 1.0),
        ( 0,  1, 1.0),

        (-1, -1, math.sqrt(2.0)),
        (-1,  1, math.sqrt(2.0)),
        ( 1, -1, math.sqrt(2.0)),
        ( 1,  1, math.sqrt(2.0)),
    ]

    visited = set()

    while open_heap:

        _, cur = heapq.heappop(open_heap)

        if cur in visited:
            continue

        visited.add(cur)

        if cur == g:
            break

        for dx, dy, cost in moves:

            nxt = (cur[0] + dx, cur[1] + dy)

            wp = grid_to_world(nxt)

            if not is_point_free(wp, obstacles, margin):
                continue

            ng = gcost[cur] + cost

            if ng < gcost.get(nxt, 1e20):

                gcost[nxt] = ng
                came[nxt] = cur

                h = math.hypot(
                    nxt[0] - g[0],
                    nxt[1] - g[1]
                )

                heapq.heappush(
                    open_heap,
                    (ng + h, nxt)
                )

    if g not in came and g != s:
        return np.linspace(start, goal, 100)

    path = [g]

    while path[-1] != s:
        path.append(came[path[-1]])

    path = path[::-1]

    pts = np.array(
        [grid_to_world(x) for x in path],
        dtype=float
    )

    return pts


def resample_polyline(points, num):
    points = np.asarray(points, dtype=float)

    seg = np.linalg.norm(
        np.diff(points, axis=0),
        axis=1
    )

    s = np.concatenate([[0.0], np.cumsum(seg)])

    if s[-1] < 1e-12:
        return np.repeat(points[:1], num, axis=0)

    target = np.linspace(0.0, s[-1], num)

    out = []

    for z in target:

        k = np.searchsorted(s, z) - 1
        k = np.clip(k, 0, len(points) - 2)

        den = max(s[k + 1] - s[k], 1e-12)

        u = (z - s[k]) / den

        out.append(
            (1.0 - u) * points[k] +
            u * points[k + 1]
        )

    return np.asarray(out)


# ============================
# 7. CAGE GENERATION
# ============================

def cage_points(center):

    # Near-closed circular cage.
    # Start and end leave only a small geometric gap.

    start_angle = np.pi + GAP / 2.0

    angles = (
        start_angle
        + np.arange(N_CAGE + 1) * CAGE_DELTA
    )

    pts = np.column_stack([
        center[0] + CAGE_RADIUS * np.cos(angles),
        center[1] + CAGE_RADIUS * np.sin(angles)
    ])

    return pts


# ============================
# 8. FIXED-LENGTH STEM INITIALIZATION
# ============================

def quadratic_curve(base, end, n_links, sign=-1.0):

    base = np.asarray(base, float)
    end = np.asarray(end, float)

    direct = end - base
    d = np.linalg.norm(direct)

    total = n_links * L

    if d > total:
        raise ValueError(
            f"Stem target unreachable: distance={d:.3f}, "
            f"available={total:.3f}"
        )

    if d < 1e-10:
        direct = np.array([1.0, 0.0])
        d = 1.0

    unit = direct / d

    perp = np.array([-unit[1], unit[0]])

    midpoint = (base + end) / 2.0

    def curve_with_amp(amp):

        control = midpoint + sign * amp * perp

        t = np.linspace(0.0, 1.0, 1000)

        pts = (
            ((1.0 - t) ** 2)[:, None] * base
            + (2.0 * (1.0 - t) * t)[:, None] * control
            + (t ** 2)[:, None] * end
        )

        length = np.sum(
            np.linalg.norm(
                np.diff(pts, axis=0),
                axis=1
            )
        )

        return pts, length

    lo = 0.0
    hi = 8.0

    for _ in range(50):

        mid = (lo + hi) / 2.0

        _, length = curve_with_amp(mid)

        if length < total:
            lo = mid
        else:
            hi = mid

    pts, _ = curve_with_amp(
        (lo + hi) / 2.0
    )

    return resample_polyline(
        pts,
        n_links + 1
    )


# ============================
# 9. LINK-LENGTH PROJECTION
# ============================

def project_lengths_base(points, iterations=8):

    p = points.copy()

    base = CFG["BASE"].copy()

    for _ in range(iterations):

        p[0] = base

        # forward
        for i in range(1, len(p)):

            v = p[i] - p[i - 1]
            n = np.linalg.norm(v)

            if n < 1e-12:
                v = np.array([1.0, 0.0])
                n = 1.0

            p[i] = (
                p[i - 1]
                + L * v / n
            )

        p[0] = base

    return p


def project_stem_endpoints(points, start, end, iterations=10):

    p = points.copy()

    for _ in range(iterations):

        # backward from end
        p[-1] = end

        for i in range(len(p) - 2, -1, -1):

            v = p[i] - p[i + 1]
            n = np.linalg.norm(v)

            if n < 1e-12:
                v = np.array([-1.0, 0.0])
                n = 1.0

            p[i] = (
                p[i + 1]
                + L * v / n
            )

        # forward from base
        p[0] = start

        for i in range(1, len(p)):

            v = p[i] - p[i - 1]
            n = np.linalg.norm(v)

            if n < 1e-12:
                v = np.array([1.0, 0.0])
                n = 1.0

            p[i] = (
                p[i - 1]
                + L * v / n
            )

    return p


# ============================
# 10. STEM OBSTACLE REFINEMENT
# ============================

def refine_stem(stem, obstacles, end_target):

    p = stem.copy()

    margin = (
        CFG["ROBOT_RADIUS"]
        + CFG["D_SAFE"]
    )

    for _ in range(CFG["SHAPE_REFINE_ITERS"]):

        newp = p.copy()

        # smoothness force
        for i in range(1, len(p) - 1):

            smooth_target = (
                p[i - 1] + p[i + 1]
            ) / 2.0

            newp[i] += (
                0.05
                * (smooth_target - p[i])
            )

        # obstacle repulsion
        for i in range(1, len(p) - 1):

            for rect in obstacles:

                d = point_rect_signed_distance(
                    p[i],
                    rect
                )

                if d < margin:

                    direction = point_rect_escape_vector(
                        p[i],
                        rect
                    )

                    strength = (
                        margin - d
                    )

                    newp[i] += (
                        0.42
                        * strength
                        * direction
                    )

        # self-spacing
        for i in range(1, len(p) - 1):

            for j in range(i + 4, len(p) - 1):

                v = newp[i] - newp[j]
                dist = np.linalg.norm(v)

                if dist < CFG["D_SELF"] * 2.0:

                    if dist < 1e-8:
                        direction = np.array(
                            [0.0, 1.0]
                        )
                    else:
                        direction = v / dist

                    push = (
                        CFG["D_SELF"] * 2.0
                        - dist
                    ) * 0.08

                    newp[i] += push * direction
                    newp[j] -= push * direction

        p = project_stem_endpoints(
            newp,
            CFG["BASE"],
            end_target,
            iterations=3
        )

    return p


# ============================
# 11. EXPERT BACKBONE GENERATOR
# ============================

def generate_expert_backbone(
    object_center,
    obstacles,
    bulge_sign=-1.0
):

    cage = cage_points(object_center)

    cage_start = cage[0]

    stem = quadratic_curve(
        CFG["BASE"],
        cage_start,
        N_STEM,
        sign=bulge_sign
    )

    stem = refine_stem(
        stem,
        obstacles,
        cage_start
    )

    # stem includes cage_start
    # cage also includes cage_start
    full = np.vstack([
        stem,
        cage[1:]
    ])

    assert len(full) == N + 1

    return full


# ============================
# 12. GEOMETRIC FEATURES
# ============================

def nearest_obstacle_vector(p, obstacles):

    best_d = 1e10
    best_v = np.zeros(2)

    for rect in obstacles:

        center = np.array(
            [rect[0], rect[1]],
            dtype=float
        )

        v = center - p
        d = np.linalg.norm(v)

        if d < best_d:
            best_d = d
            best_v = v

    return best_v


def make_token_features(
    backbone,
    object_center,
    goal,
    obstacles
):

    feats = []

    for i, p in enumerate(backbone):

        obs_vec = nearest_obstacle_vector(
            p,
            obstacles
        )

        idx = i / N

        feats.append([
            p[0] / 5.0,
            p[1] / 4.0,

            (object_center[0] - p[0]) / 5.0,
            (object_center[1] - p[1]) / 4.0,

            (goal[0] - p[0]) / 5.0,
            (goal[1] - p[1]) / 4.0,

            obs_vec[0] / 5.0,
            obs_vec[1] / 4.0,

            idx
        ])

    return np.asarray(
        feats,
        dtype=np.float32
    )


# ============================
# 13. TRANSFORMER DATASET
# ============================

class ShapeDataset(Dataset):

    def __init__(self):

        self.x = []
        self.y = []
        self.base_xy = []

        for scene_id, scene in enumerate(SCENES):

            static_obs = scene["obstacles"]

            coarse_path = astar_path(
                scene["object_start"],
                scene["goal"],
                static_obs,
                margin=(
                    CAGE_RADIUS
                    + CFG["OBJECT_RADIUS"]
                    + CFG["D_SAFE"]
                )
            )

            samples = resample_polyline(
                coarse_path,
                24
            )

            for center in samples:

                # use static environment for training
                obs = list(static_obs)

                try:
                    target = generate_expert_backbone(
                        center,
                        obs,
                        bulge_sign=-1.0
                    )
                except Exception:
                    continue

                for _ in range(2):

                    noisy = target.copy()

                    noise = np.random.normal(
                        0.0,
                        0.055,
                        size=noisy.shape
                    )

                    noise[0] = 0.0

                    noisy += noise

                    feat = make_token_features(
                        noisy,
                        center,
                        scene["goal"],
                        obs
                    )

                    residual = (
                        target - noisy
                    ).astype(np.float32)

                    self.x.append(feat)

                    self.y.append(residual)

                    self.base_xy.append(
                        noisy.astype(np.float32)
                    )

        self.x = torch.tensor(
            np.stack(self.x),
            dtype=torch.float32
        )

        self.y = torch.tensor(
            np.stack(self.y),
            dtype=torch.float32
        )

        self.base_xy = torch.tensor(
            np.stack(self.base_xy),
            dtype=torch.float32
        )

        print(
            "Transformer training samples:",
            len(self.x)
        )

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):

        return (
            self.x[i],
            self.y[i],
            self.base_xy[i]
        )


# ============================
# 14. TRANSFORMER MODEL
# ============================

class ShapeTransformer(nn.Module):

    def __init__(
        self,
        input_dim=9,
        d_model=64,
        nhead=4,
        layers=2,
        ff=128,
        dropout=0.05,
        n_tokens=51
    ):

        super().__init__()

        self.embed = nn.Linear(
            input_dim,
            d_model
        )

        self.pos = nn.Parameter(
            torch.zeros(
                1,
                n_tokens,
                d_model
            )
        )

        nn.init.normal_(
            self.pos,
            std=0.02
        )

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2)
        )

    def forward(self, x):

        z = self.embed(x)
        z = z + self.pos[:, :x.shape[1]]

        h = self.encoder(z)

        delta = self.head(h)

        # bound residual correction
        return 0.22 * torch.tanh(delta)


# ============================
# 15. TRAIN TRANSFORMER
# ============================

dataset = ShapeDataset()

loader = DataLoader(
    dataset,
    batch_size=CFG["TRANSFORMER_BATCH_SIZE"],
    shuffle=True
)

transformer = ShapeTransformer(
    d_model=CFG["D_MODEL"],
    nhead=CFG["NHEAD"],
    layers=CFG["NUM_ENCODER_LAYERS"],
    ff=CFG["DIM_FEEDFORWARD"],
    dropout=CFG["DROPOUT"],
    n_tokens=N + 1
).to(DEVICE)

optT = torch.optim.AdamW(
    transformer.parameters(),
    lr=CFG["TRANSFORMER_LR"]
)

loss_fn = nn.MSELoss()

print("\nTraining Transformer...")

for epoch in range(
    CFG["TRANSFORMER_EPOCHS"]
):

    transformer.train()

    losses = []

    for xb, yb, baseb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        pred = transformer(xb)

        loss = loss_fn(
            pred,
            yb
        )

        optT.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            transformer.parameters(),
            1.0
        )

        optT.step()

        losses.append(
            loss.item()
        )

    if (
        epoch == 0
        or (epoch + 1) % 5 == 0
        or epoch == CFG["TRANSFORMER_EPOCHS"] - 1
    ):

        print(
            f"Transformer epoch "
            f"{epoch+1:02d}/"
            f"{CFG['TRANSFORMER_EPOCHS']} "
            f"loss={np.mean(losses):.6f}"
        )


# ============================
# 16. NEURAL SHAPE CONTROLLER
# ============================

class NeuralShapeController(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(7, 48),
            nn.GELU(),
            nn.Linear(48, 48),
            nn.GELU(),
            nn.Linear(48, 2)
        )

    def forward(self, x):
        return self.net(x)


controller = NeuralShapeController().to(DEVICE)


# ============================
# 17. SYNTHETIC CONTROLLER DATA
# ============================

num_ctrl = 30000

cur = np.random.uniform(
    [-5.0, -4.0],
    [ 5.0,  4.0],
    size=(num_ctrl, 2)
)

err = np.random.normal(
    0.0,
    0.45,
    size=(num_ctrl, 2)
)

target = cur + err

idx = np.random.uniform(
    0.0,
    1.0,
    size=(num_ctrl, 1)
)

feat_ctrl = np.hstack([
    cur / np.array([5.0, 4.0]),
    target / np.array([5.0, 4.0]),
    err / np.array([1.0, 1.0]),
    idx
]).astype(np.float32)

target_ctrl = (
    CFG["CONTROL_GAIN"] * err
).astype(np.float32)

ctrl_ds = torch.utils.data.TensorDataset(
    torch.tensor(feat_ctrl),
    torch.tensor(target_ctrl)
)

ctrl_loader = DataLoader(
    ctrl_ds,
    batch_size=CFG["NSC_BATCH_SIZE"],
    shuffle=True
)

optC = torch.optim.Adam(
    controller.parameters(),
    lr=CFG["NSC_LR"]
)

print("\nTraining Neural Shape Controller...")

for epoch in range(
    CFG["NSC_EPOCHS"]
):

    losses = []

    for xb, yb in ctrl_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        pred = controller(xb)

        loss = loss_fn(pred, yb)

        optC.zero_grad()
        loss.backward()
        optC.step()

        losses.append(
            loss.item()
        )

    if (
        epoch == 0
        or (epoch + 1) % 5 == 0
        or epoch == CFG["NSC_EPOCHS"] - 1
    ):

        print(
            f"NSC epoch "
            f"{epoch+1:02d}/"
            f"{CFG['NSC_EPOCHS']} "
            f"loss={np.mean(losses):.6f}"
        )


# ============================
# 18. TRANSFORMER INFERENCE
# ============================

@torch.no_grad()
def transformer_predict(
    coarse_backbone,
    object_center,
    goal,
    obstacles
):

    transformer.eval()

    feat = make_token_features(
        coarse_backbone,
        object_center,
        goal,
        obstacles
    )

    xt = torch.tensor(
        feat[None],
        dtype=torch.float32,
        device=DEVICE
    )

    residual = (
        transformer(xt)
        .cpu()
        .numpy()[0]
    )

    pred = (
        coarse_backbone
        + residual
    )

    pred[0] = CFG["BASE"]

    return pred


# ============================
# 19. SYMMETRY-PRESERVING OPTIMIZATION
# ============================

def optimize_shape(
    predicted,
    object_center,
    obstacles
):

    p = predicted.copy()

    desired_cage = cage_points(
        object_center
    )

    # distal start index
    cage_start_idx = N_STEM

    # force base
    p[0] = CFG["BASE"]

    for _ in range(
        CFG["SHAPE_REFINE_ITERS"]
    ):

        # --------------------------------
        # 1. Symmetry / cage attraction
        # --------------------------------
        distal = p[
            cage_start_idx:
        ]

        p[
            cage_start_idx:
        ] += (
            0.62
            * (
                desired_cage
                - distal
            )
        )

        # --------------------------------
        # 2. Smooth stem
        # --------------------------------
        for i in range(
            1,
            cage_start_idx
        ):

            if i < cage_start_idx - 1:

                mid = (
                    p[i - 1]
                    + p[i + 1]
                ) / 2.0

                p[i] += (
                    0.045
                    * (mid - p[i])
                )

        # --------------------------------
        # 3. Obstacle repulsion
        # --------------------------------
        margin = (
            CFG["D_SAFE"]
            + CFG["ROBOT_RADIUS"]
        )

        for i in range(
            1,
            cage_start_idx
        ):

            for rect in obstacles:

                d = point_rect_signed_distance(
                    p[i],
                    rect
                )

                if d < margin:

                    direction = point_rect_escape_vector(
                        p[i],
                        rect
                    )

                    p[i] += (
                        0.34
                        * (margin - d)
                        * direction
                    )

        # --------------------------------
        # 4. Self-repulsion
        # --------------------------------
        for i in range(
            1,
            len(p) - 1
        ):

            for j in range(
                i + 4,
                len(p) - 1
            ):

                v = p[i] - p[j]
                dist = np.linalg.norm(v)

                threshold = (
                    CFG["D_SELF"] * 1.7
                )

                if dist < threshold:

                    if dist < 1e-8:
                        direction = np.array(
                            [0.0, -1.0]
                        )
                    else:
                        direction = v / dist

                    push = (
                        threshold - dist
                    ) * 0.055

                    p[i] += (
                        push * direction
                    )

                    p[j] -= (
                        push * direction
                    )

        # --------------------------------
        # 5. Rebuild exact stem
        # --------------------------------
        cage_start = desired_cage[0]

        stem = p[
            :cage_start_idx + 1
        ].copy()

        stem = project_stem_endpoints(
            stem,
            CFG["BASE"],
            cage_start,
            iterations=3
        )

        p[
            :cage_start_idx + 1
        ] = stem

        # exact cage hard projection
        p[
            cage_start_idx:
        ] = desired_cage

    p[0] = CFG["BASE"]

    return p


# ============================
# 20. NEURAL CONTROLLER STEP
# ============================

@torch.no_grad()
def neural_control_step(
    current,
    target
):

    controller.eval()

    feats = []

    for i in range(N + 1):

        err = target[i] - current[i]

        feats.append([
            current[i, 0] / 5.0,
            current[i, 1] / 4.0,

            target[i, 0] / 5.0,
            target[i, 1] / 4.0,

            err[0],
            err[1],

            i / N
        ])

    x = torch.tensor(
        np.asarray(feats),
        dtype=torch.float32,
        device=DEVICE
    )

    correction = (
        controller(x)
        .cpu()
        .numpy()
    )

    norms = np.linalg.norm(
        correction,
        axis=1
    )

    scale = np.ones_like(norms)

    mask = norms > CFG["MAX_POINT_STEP"]

    scale[mask] = (
        CFG["MAX_POINT_STEP"]
        / norms[mask]
    )

    correction *= scale[:, None]

    newp = (
        current + correction
    )

    newp[0] = CFG["BASE"]

    # strong distal cage tracking
    newp[N_STEM:] = (
        0.18 * newp[N_STEM:]
        + 0.82 * target[N_STEM:]
    )

    # retain link lengths along stem
    stem = newp[:N_STEM + 1]

    stem = project_stem_endpoints(
        stem,
        CFG["BASE"],
        target[N_STEM],
        iterations=CFG["ACTUAL_CONSTRAINT_ITERS"]
    )

    newp[:N_STEM + 1] = stem

    # cage is already exact fixed-length geometry
    newp[N_STEM:] = (
        0.10 * newp[N_STEM:]
        + 0.90 * target[N_STEM:]
    )

    return newp


# ============================
# 21. CAGING METRICS
# ============================

def cage_metrics(
    robot_points,
    object_center
):

    cage = robot_points[
        N_STEM:
    ]

    cage_center = np.mean(
        cage,
        axis=0
    )

    center_error = np.linalg.norm(
        cage_center - object_center
    )

    radial = np.linalg.norm(
        cage - object_center,
        axis=1
    )

    radial_error = np.mean(
        np.abs(
            radial - CAGE_RADIUS
        )
    )

    # endpoint cage gap
    gap_dist = np.linalg.norm(
        cage[0] - cage[-1]
    )

    # geometric requirement:
    # gap must be smaller than object diameter
    gap_ok = (
        gap_dist
        < 2.0 * CFG["OBJECT_RADIUS"] * 0.95
    )

    center_ok = (
        center_error < 0.22
    )

    radial_ok = (
        radial_error < 0.16
    )

    caged = bool(
        gap_ok
        and center_ok
        and radial_ok
    )

    return (
        caged,
        float(center_error),
        float(radial_error),
        float(gap_dist)
    )


# ============================
# 22. BACKBONE SHAPE ERROR
# ============================

def mean_shape_error(
    actual,
    desired
):

    return float(
        np.mean(
            np.linalg.norm(
                actual - desired,
                axis=1
            )
        )
    )


# ============================
# 23. PLAN OBJECT PATH
# ============================

def plan_scene_path(scene):

    static_obs = scene["obstacles"]

    margin = (
        CAGE_RADIUS
        + CFG["OBJECT_RADIUS"]
        + CFG["D_SAFE"]
        + CFG["OBJECT_PATH_MARGIN"]
    )

    raw = astar_path(
        scene["object_start"],
        scene["goal"],
        static_obs,
        margin=margin
    )

    path = resample_polyline(
        raw,
        CFG["TRANSPORT_STEPS"]
    )

    path[0] = scene["object_start"]
    path[-1] = scene["goal"]

    return path


# ============================
# 24. INITIAL ROBOT SHAPE
# ============================

def initialize_robot(scene):

    obs = list(
        scene["obstacles"]
    )

    target = generate_expert_backbone(
        scene["object_start"],
        obs,
        bulge_sign=-1.0
    )

    # intentionally slightly perturbed
    current = target.copy()

    current[
        1:N_STEM
    ] += np.random.normal(
        0,
        0.025,
        size=(N_STEM - 1, 2)
    )

    current[0] = CFG["BASE"]

    return current


# ============================
# 25. SINGLE SCENE SIMULATION
# ============================

def simulate_scene(
    scene_id,
    scene
):

    start_time = time.time()

    print(
        f"\nRunning scene "
        f"{scene_id + 1}/"
        f"{CFG['N_SCENES']} ..."
    )

    object_path = plan_scene_path(
        scene
    )

    object_idx = 0

    object_center = (
        object_path[0].copy()
    )

    robot = initialize_robot(
        scene
    )

    # records
    video_frames = []

    shape_errors = []
    center_errors = []
    radial_errors = []
    cage_flags = []

    robot_collision_flags = []
    object_collision_flags = []

    robot_clearances = []
    object_clearances = []

    curvatures = []

    singularity_events = 0

    prev_singular = False

    total_frame = 0

    # ------------------------------------
    # Main loop
    # ------------------------------------
    while total_frame < CFG["MAX_TOTAL_FRAMES"]:

        mov_obs = moving_rect(
            scene,
            total_frame
        )

        obstacles_now = (
            list(scene["obstacles"])
            + [mov_obs]
        )

        # ------------------------------------
        # Expert coarse backbone
        # ------------------------------------
        coarse = generate_expert_backbone(
            object_center,
            obstacles_now,
            bulge_sign=-1.0
        )

        # ------------------------------------
        # Transformer prediction
        # ------------------------------------
        predicted = transformer_predict(
            coarse,
            object_center,
            scene["goal"],
            obstacles_now
        )

        # ------------------------------------
        # Symmetry-preserving optimization
        # ------------------------------------
        optimized = optimize_shape(
            predicted,
            object_center,
            obstacles_now
        )

        # ------------------------------------
        # Neural Shape Controller
        # ------------------------------------
        robot = neural_control_step(
            robot,
            optimized
        )

        # ------------------------------------
        # Metrics
        # ------------------------------------
        caged, c_err, r_err, cage_gap = (
            cage_metrics(
                robot,
                object_center
            )
        )

        robot_clear = (
            min_link_obstacle_clearance(
                robot,
                obstacles_now
            )
            - CFG["ROBOT_RADIUS"]
        )

        obj_clear = (
            object_obstacle_clearance(
                object_center,
                CFG["OBJECT_RADIUS"],
                obstacles_now
            )
        )

        robot_collision = (
            robot_clear < 0.0
        )

        object_collision = (
            obj_clear < 0.0
        )

        curv = max_backbone_curvature(
            robot
        )

        self_int = self_intersections(
            robot
        )

        # near singularity:
        singular = bool(
            self_int > 0
            or curv > 2.45
        )

        if singular and not prev_singular:
            singularity_events += 1

        prev_singular = singular

        # ------------------------------------
        # Object movement logic
        #
        # IMPORTANT:
        # object moves ONLY if cage is valid
        # and object/robot are collision-free.
        # ------------------------------------
        if (
            total_frame >= CFG["WARMUP_FRAMES"]
            and caged
            and not robot_collision
            and not object_collision
        ):

            if object_idx < len(object_path) - 1:

                candidate_idx = min(
                    object_idx + 1,
                    len(object_path) - 1
                )

                candidate = (
                    object_path[candidate_idx]
                )

                # dynamic obstacle safety
                candidate_clear = (
                    object_obstacle_clearance(
                        candidate,
                        CFG["OBJECT_RADIUS"],
                        obstacles_now
                    )
                )

                if candidate_clear > CFG["D_SAFE"]:

                    object_idx = candidate_idx
                    object_center = candidate.copy()

        # ------------------------------------
        # Evaluation begins after warmup
        # ------------------------------------
        if total_frame >= CFG["WARMUP_FRAMES"]:

            shape_errors.append(
                mean_shape_error(
                    robot,
                    optimized
                )
            )

            center_errors.append(
                c_err
            )

            radial_errors.append(
                r_err
            )

            cage_flags.append(
                float(caged)
            )

            robot_collision_flags.append(
                float(robot_collision)
            )

            object_collision_flags.append(
                float(object_collision)
            )

            robot_clearances.append(
                robot_clear
            )

            object_clearances.append(
                obj_clear
            )

            curvatures.append(
                curv
            )

        # ------------------------------------
        # Save frame data
        # ------------------------------------
        video_frames.append({

            "robot": robot.copy(),

            "desired": optimized.copy(),

            "object": object_center.copy(),

            "goal": scene["goal"].copy(),

            "static_obs": list(
                scene["obstacles"]
            ),

            "moving_obs": mov_obs,

            "caged": caged,

            "frame": total_frame,

            "path": object_path.copy(),
        })

        # ------------------------------------
        # stop if goal reached and stable
        # ------------------------------------
        goal_error = np.linalg.norm(
            object_center - scene["goal"]
        )

        if (
            object_idx >= len(object_path) - 1
            and goal_error < 0.08
        ):

            # keep several stable final frames
            if total_frame > (
                CFG["WARMUP_FRAMES"]
                + CFG["TRANSPORT_STEPS"]
            ):
                break

        total_frame += 1

    runtime = time.time() - start_time

    final_goal_error = float(
        np.linalg.norm(
            object_center
            - scene["goal"]
        )
    )

    cage_success_rate = (
        float(np.mean(cage_flags))
        if cage_flags
        else 0.0
    )

    collision_rate_robot = (
        float(
            np.mean(robot_collision_flags)
        )
        if robot_collision_flags
        else 1.0
    )

    collision_rate_object = (
        float(
            np.mean(object_collision_flags)
        )
        if object_collision_flags
        else 1.0
    )

    min_clear_robot = (
        float(
            np.min(robot_clearances)
        )
        if robot_clearances
        else np.nan
    )

    min_clear_object = (
        float(
            np.min(object_clearances)
        )
        if object_clearances
        else np.nan
    )

    max_curvature = (
        float(np.max(curvatures))
        if curvatures
        else np.nan
    )

    result = {

        "scene": scene_id + 1,

        "success": int(
            final_goal_error < 0.12
            and cage_success_rate >= 0.85
            and collision_rate_robot <= 0.01
            and collision_rate_object <= 0.001
            and min_clear_robot >= -0.01
            and singularity_events == 0
        ),

        "final_goal_error":
            final_goal_error,

        "mean_shape_error":
            float(np.mean(shape_errors))
            if shape_errors
            else np.nan,

        "mean_cage_center_error":
            float(np.mean(center_errors))
            if center_errors
            else np.nan,

        "mean_cage_radial_error":
            float(np.mean(radial_errors))
            if radial_errors
            else np.nan,

        "cage_success_rate":
            cage_success_rate,

        "collision_rate_robot":
            collision_rate_robot,

        "collision_rate_object":
            collision_rate_object,

        "min_clearance_robot":
            min_clear_robot,

        "min_clearance_object":
            min_clear_object,

        "max_curvature":
            max_curvature,

        "singularity_events":
            singularity_events,

        "runtime_sec":
            runtime,
    }

    print(result)

    return (
        result,
        video_frames
    )


# ============================
# 26. VIDEO SAVER
# ============================

def save_scene_video(
    scene_id,
    scene,
    frames
):

    if not CFG["SAVE_VIDEO"]:
        return None

    path = os.path.join(
        OUTPUT_DIR,
        f"scene_{scene_id}_hyper_redundant_caging.mp4"
    )

    fig, ax = plt.subplots(
        figsize=(9.5, 7.0)
    )

    writer = FFMpegWriter(
        fps=CFG["FPS"],
        metadata={
            "title":
            f"Hyper-Redundant Robot Scene {scene_id}"
        },
        bitrate=1800
    )

    with writer.saving(
        fig,
        path,
        CFG["VIDEO_DPI"]
    ):

        for data in frames:

            ax.clear()

            ax.set_xlim(
                CFG["X_MIN"],
                CFG["X_MAX"]
            )

            ax.set_ylim(
                CFG["Y_MIN"],
                CFG["Y_MAX"]
            )

            ax.set_aspect(
                "equal",
                adjustable="box"
            )

            ax.grid(
                alpha=0.18
            )

            # static obstacles
            for rect in data["static_obs"]:

                cx, cy, w, h = rect

                ax.add_patch(
                    Rectangle(
                        (
                            cx - w / 2.0,
                            cy - h / 2.0
                        ),
                        w,
                        h,
                        facecolor="0.72",
                        edgecolor="black",
                        linewidth=1.2
                    )
                )

            # moving obstacle
            cx, cy, w, h = (
                data["moving_obs"]
            )

            ax.add_patch(
                Rectangle(
                    (
                        cx - w / 2.0,
                        cy - h / 2.0
                    ),
                    w,
                    h,
                    facecolor="0.90",
                    edgecolor="black",
                    linewidth=1.5,
                    hatch="//"
                )
            )

            # planned object path
            pth = data["path"]

            ax.plot(
                pth[:, 0],
                pth[:, 1],
                "--",
                color="0.45",
                linewidth=1.1,
                label="Object path"
            )

            # desired backbone
            desired = data["desired"]

            ax.plot(
                desired[:, 0],
                desired[:, 1],
                "--",
                color="0.45",
                linewidth=1.0,
                label="Desired backbone"
            )

            # actual robot
            robot = data["robot"]

            ax.plot(
                robot[:, 0],
                robot[:, 1],
                "-o",
                color="black",
                markersize=2.6,
                linewidth=1.45,
                label="50-link robot"
            )

            # base
            ax.plot(
                robot[0, 0],
                robot[0, 1],
                "ks",
                markersize=7,
                label="Robot base"
            )

            # object
            obj = data["object"]

            ax.add_patch(
                Circle(
                    obj,
                    CFG["OBJECT_RADIUS"],
                    facecolor="white",
                    edgecolor="black",
                    linewidth=2.0
                )
            )

            ax.text(
                obj[0],
                obj[1],
                "Object",
                ha="center",
                va="center",
                fontsize=8
            )

            # goal
            goal = data["goal"]

            ax.plot(
                goal[0],
                goal[1],
                marker="*",
                color="black",
                markersize=13,
                label="Goal"
            )

            status = (
                "CAGED"
                if data["caged"]
                else "FORMING CAGE"
            )

            ax.set_title(
                f"{scene['name']} | "
                f"Frame {data['frame']} | "
                f"{status}"
            )

            ax.set_xlabel("x")
            ax.set_ylabel("y")

            ax.legend(
                loc="upper left",
                fontsize=7,
                frameon=True
            )

            writer.grab_frame()

    plt.close(fig)

    print("Saved:", path)

    return path


# ============================
# 27. RUN ALL 5 SCENES
# ============================

all_results = []

video_paths = []

for scene_id, scene in enumerate(SCENES):

    result, frame_data = (
        simulate_scene(
            scene_id,
            scene
        )
    )

    all_results.append(
        result
    )

    video_path = (
        save_scene_video(
            scene_id + 1,
            scene,
            frame_data
        )
    )

    video_paths.append(
        video_path
    )


# ============================
# 28. RESULTS DATAFRAME
# ============================

results_df = pd.DataFrame(
    all_results
)

print("\n")
print("=" * 120)
print("FINAL SCENE RESULTS")
print("=" * 120)

display(results_df)


# ============================
# 29. MEAN ± STD SUMMARY
# ============================

metric_cols = [

    "final_goal_error",
    "mean_shape_error",
    "mean_cage_center_error",
    "mean_cage_radial_error",

    "cage_success_rate",

    "collision_rate_robot",
    "collision_rate_object",

    "min_clearance_robot",
    "min_clearance_object",

    "max_curvature",

    "singularity_events",
    "runtime_sec",
]

summary_rows = []

for col in metric_cols:

    vals = pd.to_numeric(
        results_df[col],
        errors="coerce"
    )

    summary_rows.append({

        "metric": col,

        "mean":
            float(vals.mean()),

        "std":
            float(vals.std(ddof=1)),

        "min":
            float(vals.min()),

        "max":
            float(vals.max()),

        "mean_std":
            f"{vals.mean():.6f} ± "
            f"{vals.std(ddof=1):.6f}",
    })

summary_df = pd.DataFrame(
    summary_rows
)

print("\n")
print("=" * 120)
print("MEAN ± STANDARD DEVIATION")
print("=" * 120)

display(summary_df)


# ============================
# 30. OVERALL SUCCESS SUMMARY
# ============================

overall_success_rate = (
    results_df["success"].mean()
)

print(
    "\nSuccessful scenes:",
    int(results_df["success"].sum()),
    "/",
    len(results_df)
)

print(
    "Overall scene success rate:",
    f"{overall_success_rate:.2%}"
)


# ============================
# 31. SAVE RESULT TABLES
# ============================

results_path = os.path.join(
    OUTPUT_DIR,
    "scene_results.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "summary_mean_std.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("\nSaved:", results_path)
print("Saved:", summary_path)


# ============================
# 32. TABLE FOR MANUSCRIPT
# ============================

paper_table = results_df[
    [
        "scene",
        "success",
        "final_goal_error",
        "mean_shape_error",
        "mean_cage_center_error",
        "mean_cage_radial_error",
        "cage_success_rate",
        "collision_rate_robot",
        "collision_rate_object",
        "min_clearance_robot",
        "min_clearance_object",
        "max_curvature",
        "singularity_events",
        "runtime_sec"
    ]
].copy()

paper_table.columns = [

    "Scene",
    "Success",
    "Final Goal Error",
    "Mean Shape Error",
    "Mean Cage Center Error",
    "Mean Cage Radial Error",
    "Cage Success Rate",
    "Robot Collision Rate",
    "Object Collision Rate",
    "Min Robot Clearance",
    "Min Object Clearance",
    "Max Curvature",
    "Singularity Events",
    "Runtime (s)"
]

paper_table_path = os.path.join(
    OUTPUT_DIR,
    "Table_Experimental_Results.csv"
)

paper_table.to_csv(
    paper_table_path,
    index=False
)

print(
    "\nPaper-ready result table:"
)

display(paper_table)

print(
    "Saved:",
    paper_table_path
)


# ============================
# 33. TABLE 3:
# IMPLEMENTATION & HYPERPARAMETERS
# ============================

table3_rows = [

    [
        "Robot",
        "Robot type",
        "Planar 2D hyper-redundant manipulator"
    ],

    [
        "Robot",
        "Number of links",
        CFG["N_LINKS"]
    ],

    [
        "Robot",
        "Degrees of freedom",
        CFG["N_LINKS"]
    ],

    [
        "Robot",
        "Joint type",
        "Revolute"
    ],

    [
        "Robot",
        "Link length",
        CFG["LINK_LENGTH"]
    ],

    [
        "Robot",
        "Cage-forming distal links",
        CFG["N_CAGE_LINKS"]
    ],

    [
        "Robot",
        "Cage radius",
        CFG["CAGE_RADIUS"]
    ],

    [
        "Modeling",
        "Motion space",
        "Planar x-y"
    ],

    [
        "Modeling",
        "Robot model",
        "Kinematic"
    ],

    [
        "Modeling",
        "Robot-object interaction",
        "Frictionless"
    ],

    [
        "Modeling",
        "Gravity",
        "Not explicitly modeled"
    ],

    [
        "Safety",
        "Obstacle safety distance",
        CFG["D_SAFE"]
    ],

    [
        "Safety",
        "Self-clearance threshold",
        CFG["D_SELF"]
    ],

    [
        "Safety",
        "Whole-body collision checking",
        "Enabled"
    ],

    [
        "Safety",
        "Self-collision checking",
        "Enabled"
    ],

    [
        "Transformer",
        "Embedding dimension",
        CFG["D_MODEL"]
    ],

    [
        "Transformer",
        "Attention heads",
        CFG["NHEAD"]
    ],

    [
        "Transformer",
        "Encoder layers",
        CFG["NUM_ENCODER_LAYERS"]
    ],

    [
        "Transformer",
        "Feed-forward dimension",
        CFG["DIM_FEEDFORWARD"]
    ],

    [
        "Transformer",
        "Dropout",
        CFG["DROPOUT"]
    ],

    [
        "Transformer",
        "Training epochs",
        CFG["TRANSFORMER_EPOCHS"]
    ],

    [
        "Transformer",
        "Batch size",
        CFG["TRANSFORMER_BATCH_SIZE"]
    ],

    [
        "Transformer",
        "Learning rate",
        CFG["TRANSFORMER_LR"]
    ],

    [
        "Neural Shape Controller",
        "Training epochs",
        CFG["NSC_EPOCHS"]
    ],

    [
        "Neural Shape Controller",
        "Batch size",
        CFG["NSC_BATCH_SIZE"]
    ],

    [
        "Neural Shape Controller",
        "Learning rate",
        CFG["NSC_LR"]
    ],

    [
        "Neural Shape Controller",
        "Control gain",
        CFG["CONTROL_GAIN"]
    ],

    [
        "Shape Optimization",
        "Refinement iterations",
        CFG["SHAPE_REFINE_ITERS"]
    ],

    [
        "Experiment",
        "Number of scenarios",
        CFG["N_SCENES"]
    ],

    [
        "Experiment",
        "Warm-up frames",
        CFG["WARMUP_FRAMES"]
    ],

    [
        "Experiment",
        "Object path steps",
        CFG["TRANSPORT_STEPS"]
    ],

    [
        "Experiment",
        "Maximum frames",
        CFG["MAX_TOTAL_FRAMES"]
    ],

    [
        "Video",
        "Frame rate",
        CFG["FPS"]
    ],

    [
        "Implementation",
        "Programming language",
        "Python"
    ],

    [
        "Implementation",
        "Deep learning framework",
        "PyTorch"
    ],

    [
        "Implementation",
        "Numerical computation",
        "NumPy"
    ],

    [
        "Implementation",
        "Visualization",
        "Matplotlib"
    ],

    [
        "Implementation",
        "Execution platform",
        "Google Colaboratory"
    ],
]

table3 = pd.DataFrame(
    table3_rows,
    columns=[
        "Category",
        "Parameter",
        "Value"
    ]
)

table3_path = os.path.join(
    OUTPUT_DIR,
    "Table3_Implementation_Hyperparameters.csv"
)

table3.to_csv(
    table3_path,
    index=False
)

print("\n")
print("=" * 120)
print(
    "TABLE 3. IMPLEMENTATION AND "
    "HYPERPARAMETER SETTINGS"
)
print("=" * 120)

display(table3)

print(
    "Saved:",
    table3_path
)


# ============================
# 34. SIMPLE PAPER DIAGNOSTICS
# ============================

print("\n")
print("=" * 120)
print("PAPER DIAGNOSTICS")
print("=" * 120)

mean_cage = (
    results_df[
        "cage_success_rate"
    ].mean()
)

mean_robot_col = (
    results_df[
        "collision_rate_robot"
    ].mean()
)

mean_object_col = (
    results_df[
        "collision_rate_object"
    ].mean()
)

min_robot_clear = (
    results_df[
        "min_clearance_robot"
    ].min()
)

total_singular = int(
    results_df[
        "singularity_events"
    ].sum()
)

print(
    f"Mean cage success rate : "
    f"{mean_cage:.4f}"
)

print(
    f"Mean robot collision   : "
    f"{mean_robot_col:.6f}"
)

print(
    f"Mean object collision  : "
    f"{mean_object_col:.6f}"
)

print(
    f"Worst robot clearance  : "
    f"{min_robot_clear:.4f}"
)

print(
    f"Total singular events  : "
    f"{total_singular}"
)

print("\nQuality checks:")

print(
    "Cage >= 0.85:",
    "PASS"
    if mean_cage >= 0.85
    else "CHECK"
)

print(
    "Robot collision <= 1%:",
    "PASS"
    if mean_robot_col <= 0.01
    else "CHECK"
)

print(
    "Object collision <= 0.1%:",
    "PASS"
    if mean_object_col <= 0.001
    else "CHECK"
)

print(
    "Non-negative robot clearance:",
    "PASS"
    if min_robot_clear >= -0.01
    else "CHECK"
)

print(
    "No singularity events:",
    "PASS"
    if total_singular == 0
    else "CHECK"
)


# ============================
# 35. FINAL OUTPUT LIST
# ============================

print("\n")
print("=" * 120)
print("ALL OUTPUTS")
print("=" * 120)

for f in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        os.path.join(
            OUTPUT_DIR,
            f
        )
    )

print("\nDone.")